# 02 — Pipeline de Preprocesamiento ML
## Google Ads Analytics — Predicción de Rentabilidad (`Is_Profitable`)

---

**Objetivo:** Construir un Pipeline de scikit-learn reutilizable que transforme los datos crudos del dataset de Google Ads en una matriz numérica lista para entrenar un modelo de clasificación binaria.

**Variable objetivo:** `Is_Profitable` — 1 si `Sale_Amount > Cost`, 0 en caso contrario.

**Dataset:** `data/raw/GoogleAds_DataAnalytics_Sales_Uncleaned.csv` | 2 600 filas x 13 columnas


---
## Celda 1 — Configuración del Entorno

In [ ]:
# 1. Comandos Mágicos: recarga automática de módulos locales
%load_ext autoreload
%autoreload 2

# 2. Asegurar que Python encuentre la carpeta src/
import sys
sys.path.append('..')

# 3. Importaciones estándar
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn import set_config

# 4. Transformers personalizados
from src.transformers import (
    DropColumnsTransformer,
    MonetaryCleanerTransformer,
    DropHighMissingTransformer,
    OutlierCapper,
    DropZeroVarianceTransformer,
    SmartImputerTransformer,
)

print("✅ Librerías locales importadas correctamente.")

✅ Librerías locales importadas correctamente.


---
## Celda 2 — Carga de Datos y Creación de la Variable Objetivo

In [ ]:
# Cargamos los datos crudos
df_raw = pd.read_csv('../data/raw/GoogleAds_DataAnalytics_Sales_Uncleaned.csv')

print(f"Dataset cargado: {df_raw.shape[0]:,} filas x {df_raw.shape[1]} columnas")
print("\nColumnas originales:")
print(df_raw.dtypes)
print("\nPrimeras 3 filas:")
df_raw.head(3)

Dataset cargado: 2,600 filas x 13 columnas

Columnas originales:
Ad_ID                  str
Campaign_Name          str
Clicks             float64
Impressions        float64
Cost                   str
Leads              float64
Conversions        float64
Conversion Rate    float64
Sale_Amount            str
Ad_Date                str
Location               str
Device                 str
Keyword                str
dtype: object

Primeras 3 filas:


,Ad_ID,Campaign_Name,Clicks,Impressions,Cost,Leads,Conversions,Conversion Rate,Sale_Amount,Ad_Date,Location,Device,Keyword
0,A1000,DataAnalyticsCourse,104.0,4498.0,$231.88,14.0,7.0,0.058,$1892,2024-11-16,hyderabad,desktop,learn data analytics
1,A1001,DataAnalyticsCourse,173.0,5107.0,$216.84,10.0,8.0,0.046,$1679,20-11-2024,hyderabad,mobile,data analytics course
2,A1002,Data Anlytics Corse,90.0,4544.0,$203.66,26.0,9.0,NaN,$1624,2024/11/16,hyderabad,Desktop,data analitics online


In [ ]:
# CREACIÓN DE LA VARIABLE OBJETIVO: Is_Profitable
# Lógica de negocio: una campaña es rentable si Sale_Amount > Cost
# Nota: necesitamos limpiar el símbolo '$' antes de comparar

def parse_monetary(series: pd.Series) -> pd.Series:
    """Convierte una columna '$1,234.56' a float para crear la variable objetivo."""
    return (
        series.astype(str)
        .str.replace(r'[$,]', '', regex=True)
        .str.strip()
        .replace('', np.nan)
        .astype(float)
    )

df_raw['Is_Profitable'] = np.where(
    parse_monetary(df_raw['Sale_Amount']) > parse_monetary(df_raw['Cost']),
    1, 0
)

# Distribución de la variable objetivo
dist = df_raw['Is_Profitable'].value_counts(normalize=True) * 100
print("Variable Objetivo — Is_Profitable")
print(f"   1 (Rentable)     : {dist.get(1, 0):.1f}%")
print(f"   0 (No rentable)  : {dist.get(0, 0):.1f}%")
print(f"   Filas con NaN    : {df_raw['Is_Profitable'].isna().sum()} (se deja como 0 por defecto de np.where)")

Variable Objetivo — Is_Profitable
   1 (Rentable)     : 91.0%
   0 (No rentable)  : 9.0%
   Filas con NaN    : 0 (se deja como 0 por defecto de np.where)


---
## Celda 3 — Definición de Variables por Tipo

In [ ]:
# Separamos X (features) e y (target) ANTES de entrar al pipeline
y = df_raw['Is_Profitable']
X = df_raw.drop(columns=['Is_Profitable'])

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

# Definimos qué columnas van por qué ruta dentro del ColumnTransformer
# Excluimos: Ad_ID (eliminado), Ad_Date (eliminado), Cost y Sale_Amount
# (eliminados — son la fuente del target, serían Data Leakage directo)
variables_numericas = ['Clicks', 'Impressions', 'Leads', 'Conversions', 'Conversion Rate']

variables_categoricas = ['Campaign_Name', 'Location', 'Device', 'Keyword']

print("\nVariables numericas  ->", variables_numericas)
print("Variables categoricas ->", variables_categoricas)

X shape: (2600, 13)
y shape: (2600,)

Variables numericas  -> ['Clicks', 'Impressions', 'Leads', 'Conversions', 'Conversion Rate']
Variables categoricas -> ['Campaign_Name', 'Location', 'Device', 'Keyword']


---
## Celda 4 — Construcción del Pipeline Completo

In [ ]:
# 1. Ruta Numérica: Capping de outliers -> Varianza Cero -> Escalar
num_pipe = Pipeline([
    ('capper',        OutlierCapper(apply_capping=True)),      # IQR capping
    ('zero_variance', DropZeroVarianceTransformer()),           # elimina constantes
    ('scaler',        StandardScaler()),                        # normalización
])

# 2. Ruta Categórica: OneHot encoding (SmartImputer ya imputó los nulos)
cat_pipe = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

# 3. Enrutador (ColumnTransformer)
preprocesador = ColumnTransformer(
    transformers=[
        ('num', num_pipe, variables_numericas),
        ('cat', cat_pipe, variables_categoricas),
    ],
    remainder='drop'  # descarta cualquier columna no listada
)

# 4. EL SÚPER PIPELINE COMPLETO
pipeline_ml_completo = Pipeline([
    # Paso 1 — Eliminar columnas leak: IDs, fechas, columnas fuente del target
    ('drop_leaks',       DropColumnsTransformer(columns_to_drop=['Ad_ID', 'Ad_Date', 'Cost', 'Sale_Amount'])),
    # Paso 2 — Convertir columnas monetarias (ya no aplica tras drop, pero
    #          se mantiene en pipeline para casos de uso con datos parciales)
    ('monetary_clean',   MonetaryCleanerTransformer(columns=['Cost', 'Sale_Amount'])),
    # Paso 3 — Eliminar columnas con > 80% de nulos
    ('drop_high_nan',    DropHighMissingTransformer(threshold=0.80)),
    # Paso 4 — Imputación inteligente según % de nulos
    ('smart_imputer',    SmartImputerTransformer(low_threshold=0.10)),
    # Paso 5 — Preprocesamiento numérico + categórico
    ('preprocesamiento', preprocesador),
])

# Visualización del diagrama del pipeline
set_config(display="diagram")
pipeline_ml_completo

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('drop_leaks', ...), ('monetary_clean', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,columns_to_drop,"['Ad_ID', 'Ad_Date', ...]"
,columns,"['Cost', 'Sale_Amount']"
,threshold,0.8
,low_threshold,0.1
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have 

---
## Celda 5 — Ejecución y Verificación

In [ ]:
# ¡EL MOMENTO DE LA VERDAD!
print("Dimensiones originales (X):", X.shape)

X_procesado = pipeline_ml_completo.fit_transform(X)

print("Dimensiones procesadas    :", X_procesado.shape)
print(f"\nTipo de dato de salida    : {type(X_procesado)}")
print("\nPrimeras 3 filas de la matriz matemática lista para el modelo:")
print(X_procesado[:3])

Dimensiones originales (X): (2600, 13)
[SmartImputer] Simples  (<10%): ['Clicks', 'Impressions', 'Leads', 'Conversions']
[SmartImputer] Complejas (>10%): ['Conversion Rate'] (PENDIENTE -> KNN/Iterative)
Dimensiones procesadas    : (2600, 28)

Tipo de dato de salida    : <class 'numpy.ndarray'>

Primeras 3 filas de la matriz matemática lista para el modelo:
[[-1.03249184 -0.0292578  -1.00479351  0.20852793  0.74639713  0.
   0.          0.          1.          0.          0.          1.
   0.          0.          0.          0.          0.          0.
   0.          1.          0.          0.          0.          0.
   0.          0.          1.          0.        ]
 [ 1.00538826  0.67832403 -1.67422673  0.65476053 -0.07928683  0.
   0.          0.          1.          0.          0.          1.
   0.          0.          0.          0.          0.          0.
   0.          0.          1.          0.          0.          0.
   1.          0.          0.          0.        ]
 [-1.445974

---
## Celda 6 — Inspección de Variables Finales

In [ ]:
# Accedemos al paso de preprocesamiento (el ColumnTransformer)
enrutador = pipeline_ml_completo.named_steps['preprocesamiento']

# Recuperamos los nombres de columnas generados
nombres_finales = enrutador.get_feature_names_out()

print(f"Total de variables finales listas para el modelo: {len(nombres_finales)}\n")

for i, nombre in enumerate(nombres_finales):
    nombre_limpio = nombre.replace('num__', '').replace('cat__', '')
    print(f"{i+1:02d}. {nombre_limpio}")

Total de variables finales listas para el modelo: 28

01. Clicks
02. Impressions
03. Leads
04. Conversions
05. Conversion Rate
06. Campaign_Name_Data Analytcis Course
07. Campaign_Name_Data Analytics Corse
08. Campaign_Name_Data Anlytics Corse
09. Campaign_Name_DataAnalyticsCourse
10. Location_HYDERABAD
11. Location_Hyderbad
12. Location_hyderabad
13. Location_hydrebad
14. Device_DESKTOP
15. Device_Desktop
16. Device_MOBILE
17. Device_Mobile
18. Device_TABLET
19. Device_Tablet
20. Device_desktop
21. Device_mobile
22. Device_tablet
23. Keyword_analytics for data
24. Keyword_data analitics online
25. Keyword_data analytics course
26. Keyword_data anaytics training
27. Keyword_learn data analytics
28. Keyword_online data analytic


---
## Celda 7 — DataFrame Final con Nombres de Columnas

In [ ]:
# Convertimos la matriz numpy a un DataFrame legible
nombres_limpios = [
    n.replace('num__', '').replace('cat__', '') for n in nombres_finales
]

df_procesado = pd.DataFrame(X_procesado, columns=nombres_limpios)

print(f"DataFrame final: {df_procesado.shape[0]:,} filas x {df_procesado.shape[1]} columnas")
print("\nEstadísticas descriptivas de las columnas numéricas escaladas:")
df_procesado[['Clicks', 'Impressions', 'Leads', 'Conversions', 'Conversion Rate']].describe().round(3)

DataFrame final: 2,600 filas x 28 columnas

Estadísticas descriptivas de las columnas numéricas escaladas:


,Clicks,Impressions,Leads,Conversions,Conversion Rate
count,2600.000,2600.000,2600.000,2600.000,2600.000
mean,-0.000,0.000,-0.000,-0.000,0.000
std,1.000,1.000,1.000,1.000,1.000
min,-1.741,-1.770,-1.674,-1.576,-2.212
25%,-0.826,-0.866,-0.837,-0.684,-0.630
50%,0.001,-0.005,-0.001,0.209,-0.079
75%,0.828,0.866,0.836,0.655,0.471
max,1.773,1.715,1.673,1.547,2.123


In [ ]:
# Vista previa del DataFrame limpio
print("Primeras 5 filas del dataset procesado:")
df_procesado.head()

Primeras 5 filas del dataset procesado:


,Clicks,Impressions,Leads,Conversions,Conversion Rate,Campaign_Name_Data Analytcis Course,Campaign_Name_Data Analytics Corse,Campaign_Name_Data Anlytics Corse,Campaign_Name_DataAnalyticsCourse,Location_HYDERABAD,...,Device_Tablet,Device_desktop,Device_mobile,Device_tablet,Keyword_analytics for data,Keyword_data analitics online,Keyword_data analytics course,Keyword_data anaytics training,Keyword_learn data analytics,Keyword_online data analytic
0,-1.032492,-0.029258,-1.004794,0.208528,0.746397,0.0,0.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,1.005388,0.678324,-1.674227,0.654761,-0.079287,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,-1.445975,0.024188,1.003506,1.100993,-0.079287,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,0.089819,-1.554800,-0.502719,-0.237705,-0.079287,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
4,0.503302,-1.350310,1.672939,0.654761,-0.079287,0.0,1.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


---
## Resumen del Pipeline

| Paso | Transformer | Propósito |
|------|-------------|-----------|
| 1 | `DropColumnsTransformer` | Elimina `Ad_ID`, `Ad_Date`, `Cost`, `Sale_Amount` (fuentes del target — Data Leakage) |
| 2 | `MonetaryCleanerTransformer` | Convierte columnas `$X,XXX.XX` a `float` |
| 3 | `DropHighMissingTransformer` | Descarta columnas con > 80% de nulos |
| 4 | `SmartImputerTransformer` | Imputa <10% con mediana/moda; >10% con fallback temporal |
| 5a | `OutlierCapper` + `StandardScaler` | Recorta outliers IQR y normaliza variables numéricas |
| 5b | `OneHotEncoder` | Codifica variables categóricas en dummies |

**Nota:** `Conversion Rate` tenia **24% de nulos** — detectado por SmartImputer como columna compleja, imputada con mediana (pendiente mejora a KNN/Iterative Imputer).